# Modular Single Segmentation Notebook

Uses the same modular backend as the benchmark pipeline.


In [ ]:
import os
from pathlib import Path
import yaml

import torch
import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
import torchvision.transforms as T

from benchmark_models import build_adapters_from_config

CONFIG_PATH = "single_segmentation_config_full.yaml"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

DEVICE = cfg["inference"].get(
    "device",
    "cuda" if torch.cuda.is_available() else "cpu"
)

DEVICE = torch.device(DEVICE)

SAVE_DIR = Path(cfg["output"]["save_dir"])
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Using device:", DEVICE)
print("Save dir:", SAVE_DIR)


In [ ]:
# ==========================================================
# BUILD MODELS
# ==========================================================

adapters = build_adapters_from_config(
    cfg["models"],
    DEVICE
)

print(f"Loaded {len(adapters)} models:")

for adp in adapters:
    print(" -", adp.name)


In [ ]:
# ==========================================================
# CITYSCAPES COLORS
# ==========================================================

CITYSCAPES_COLORS = np.array([
    [128, 64,128],
    [244, 35,232],
    [ 70, 70, 70],
    [102,102,156],
    [190,153,153],
    [153,153,153],
    [250,170, 30],
    [220,220,  0],
    [107,142, 35],
    [152,251,152],
    [ 70,130,180],
    [220, 20, 60],
    [255,  0,  0],
    [  0,  0,142],
    [  0,  0, 70],
    [  0, 60,100],
    [  0, 80,100],
    [  0,  0,230],
    [119, 11, 32],
], dtype=np.uint8)

def colorize_mask(mask):
    return CITYSCAPES_COLORS[mask]

def overlay_segmentation(image_np, mask_rgb, alpha=0.5):
    return (
        image_np * (1 - alpha) +
        mask_rgb * alpha
    ).astype(np.uint8)


In [ ]:
# ==========================================================
# LOAD IMAGE
# ==========================================================

transform = T.ToTensor()

image_path = cfg["input"]["image_path"]

image_pil = Image.open(image_path).convert("RGB")

image_np = np.array(image_pil)

image_t = transform(image_pil).unsqueeze(0).to(DEVICE)

print("Image tensor shape:", image_t.shape)


In [ ]:
# ==========================================================
# RUN SEGMENTATION
# ==========================================================

for adp in adapters:

    print(f"\nRunning {adp.name}")

    pred, conf = adp.predict(image_t)

    pred = pred[0].detach().cpu().numpy()

    mask_rgb = colorize_mask(pred)

    overlay = overlay_segmentation(image_np, mask_rgb)

    model_dir = SAVE_DIR / adp.name.replace(" ", "_")
    model_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------------
    # SAVE MASK
    # -----------------------------------

    if cfg["visualization"]["save_mask"]:

        Image.fromarray(mask_rgb).save(
            model_dir / "mask.png"
        )

    # -----------------------------------
    # SAVE OVERLAY
    # -----------------------------------

    if cfg["visualization"]["save_overlay"]:

        Image.fromarray(overlay).save(
            model_dir / "overlay.png"
        )

    # -----------------------------------
    # SAVE PANEL
    # -----------------------------------

    if cfg["visualization"]["save_panel"]:

        fig, axs = plt.subplots(1, 3, figsize=(18, 6))

        axs[0].imshow(image_np)
        axs[0].set_title("Input")

        axs[1].imshow(mask_rgb)
        axs[1].set_title("Segmentation")

        axs[2].imshow(overlay)
        axs[2].set_title("Overlay")

        for ax in axs:
            ax.axis("off")

        plt.tight_layout()

        plt.savefig(
            model_dir / "panel.png",
            bbox_inches="tight"
        )

        plt.close()

print("\nDone.")
